# Age Estimation with VGGFace using DeepFace

This notebook implements age estimation using VGGFace architecture through the DeepFace library. The approach follows the DEX (Deep EXpectation of apparent age) methodology for research purposes.

## Overview
- Uses pretrained VGGFace model from DeepFace
- Implements a image preprocessing pipeline
- Provides batch processing capabilities the research datasets
- Includes visualization and analysis tools

## Imports

In [ ]:
from deepface import DeepFace

AttributeError: module 'tensorflow' has no attribute '__version__'

In [ ]:

from deepface.basemodels import VGGFace
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from pathlib import Path
from PIL import Image
import sys

# Check GPU availability
if tf.config.list_physical_devices('GPU'):
    print("GPU available")
else:
    print("Using CPU")

AttributeError: module 'tensorflow' has no attribute '__version__'

## Configuration and Setup

Define global configuration parameters and setup the environment for age estimation experiments.

In [ ]:
# Configuration parameters
CONFIG = {
    'target_size': (224, 224),  # VGGFace input size
    'batch_size': 32,
    'age_ranges': np.arange(0, 101),  # Ages 0-100
    'detector_backend': 'retinaface',  # Face detection method
    'data_root': './data',  # Root directory for images
    'output_dir': './results',  # Output directory for results
}

# Create output directory if it doesn't exist
Path(CONFIG['output_dir']).mkdir(exist_ok=True)

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("\nEnvironment configured successfully")

DEX age prediction functions ready


## Model Loading and Setup

Load the pretrained VGGFace model and setup the age estimation pipeline.

In [ ]:
# Load pretrained VGGFace model
print("Loading VGGFace model...")
try:
    model = VGGFace.loadModel()
    print("VGGFace model loaded successfully")
    print(f"Model input shape: {model.input_shape}")
    print(f"Model output shape: {model.output_shape}")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# Image preprocessing function
def preprocess_image(img_path, target_size=(224, 224)):
    """
    Preprocess image for VGGFace model input
    
    Args:
        img_path: Path to image file
        target_size: Target image size (width, height)
        
    Returns:
        Preprocessed image array ready for model input
    """
    try:
        # Load and resize image
        img = image.load_img(img_path, target_size=target_size)
        
        # Convert to array
        x = image.img_to_array(img)
        
        # Add batch dimension
        x = np.expand_dims(x, axis=0)
        
        # Apply VGG16 preprocessing
        x = preprocess_input(x)
        
        return x
    
    except Exception as e:
        print(f"Error preprocessing image {img_path}: {e}")
        return None

print("Image preprocessing function defined")

ImportError: DeepFace is not available. Install it before running this cell.

## Data Loading and Preparation

Functions for loading and preparing image data for batch processing.

In [ ]:
# Data loading functions
def find_image_files(root_dir, extensions=('*.jpg', '*.jpeg', '*.png', '*.bmp')):
    """
    Find all image files in directory and subdirectories
    
    Args:
        root_dir: Root directory to search
        extensions: File extensions to look for
        
    Returns:
        List of image file paths
    """
    root_path = Path(root_dir)
    image_files = []
    
    for ext in extensions:
        image_files.extend(list(root_path.rglob(ext)))
    
    return sorted(image_files)

def load_image_batch(image_paths, target_size=(224, 224)):
    """
    Load and preprocess a batch of images
    
    Args:
        image_paths: List of image file paths
        target_size: Target size for images
        
    Returns:
        Preprocessed image batch, valid paths
    """
    batch_images = []
    valid_paths = []
    
    for img_path in image_paths:
        processed_img = preprocess_image(img_path, target_size)
        if processed_img is not None:
            batch_images.append(processed_img)
            valid_paths.append(img_path)
        else:
            print(f"Skipping invalid image: {img_path}")
    
    if batch_images:
        return np.vstack(batch_images), valid_paths
    else:
        return None, []

# Find available images (placeholder - will search data directory)
print("Setting up data loading functions...")
data_root = Path(CONFIG['data_root'])
print(f"Data root directory: {data_root}")

# Placeholder for actual data loading
image_files = []
if data_root.exists():
    image_files = find_image_files(data_root)
    print(f"Found {len(image_files)} image files")
else:
    print(f"Data directory {data_root} not found - will use placeholder data")

## Age Prediction Functions

Functions for running age predictions using the VGGFace model with age estimation capabilities.

In [ ]:
# Age prediction functions (placeholder implementation)

def predict_age_from_features(features):
    """
    Convert VGGFace features to age prediction
    
    Args:
        features: VGGFace model output features
        
    Returns:
        Predicted age(s)
    """
    # TODO: Implement actual age prediction from VGGFace features
    # This would typically involve:
    # 1. Using the VGGFace features as input to an age regression model
    # 2. Or using a pre-trained age estimation head
    
    # Placeholder: random age prediction for demonstration
    batch_size = features.shape[0]
    predicted_ages = np.random.uniform(18, 70, batch_size)
    
    print(f"Generated {batch_size} age predictions (placeholder)")
    return predicted_ages

def predict_ages_batch(image_paths, batch_size=16):
    """
    Predict ages for a batch of images
    
    Args:
        image_paths: List of image file paths
        batch_size: Number of images to process at once
        
    Returns:
        DataFrame with predictions
    """
    if model is None:
        print("Model not loaded. Cannot make predictions.")
        return pd.DataFrame()
    
    results = []
    
    print(f"Processing {len(image_paths)} images in batches of {batch_size}")
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}: {len(batch_paths)} images")
        
        # Load and preprocess batch
        batch_images, valid_paths = load_image_batch(batch_paths)
        
        if batch_images is not None:
            # Extract features using VGGFace
            features = model.predict(batch_images, verbose=0)
            
            # Predict ages from features
            predicted_ages = predict_age_from_features(features)
            
            # Store results
            for path, age in zip(valid_paths, predicted_ages):
                results.append({
                    'image_path': str(path),
                    'filename': Path(path).name,
                    'predicted_age': age
                })
    
    return pd.DataFrame(results)

print("Age prediction functions defined (with placeholder implementation)")

## Results Generation

Run age predictions on the loaded dataset and gather results.

In [ ]:
# Run predictions on available images (placeholder)

# Sample test with a few images if available
if len(image_files) > 0:
    # Use first 5 images for testing
    sample_images = image_files[:min(5, len(image_files))]
    print(f"Running predictions on {len(sample_images)} sample images...")
    
    # Generate predictions
    results_df = predict_ages_batch(sample_images, batch_size=CONFIG['batch_size'])
    
    print("\nPrediction Results:")
    print(results_df.head())
    
    # Save results
    output_file = Path(CONFIG['output_dir']) / 'age_predictions.csv'
    results_df.to_csv(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
else:
    print("No images found. Please ensure image data is available.")
    # Create dummy results for demonstration
    results_df = pd.DataFrame({
        'image_path': ['sample1.jpg', 'sample2.jpg', 'sample3.jpg'],
        'filename': ['sample1.jpg', 'sample2.jpg', 'sample3.jpg'],
        'predicted_age': [25.3, 34.7, 42.1]
    })
    print("Created dummy results for demonstration")

## Data Visualization and Analysis

Visualize the results and perform statistical analysis of age predictions.

In [ ]:
# Visualization and analysis of results

def visualize_age_predictions(results_df):
    """
    Create visualizations for age prediction results
    
    Args:
        results_df: DataFrame with prediction results
    """
    if results_df.empty:
        print("No results to visualize")
        return
    
    # Filter valid predictions
    valid_results = results_df.dropna(subset=['predicted_age'])
    
    if len(valid_results) == 0:
        print("No valid predictions to visualize")
        return
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Age Prediction Analysis', fontsize=16)
    
    # Age distribution histogram
    axes[0, 0].hist(valid_results['predicted_age'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Age Distribution')
    axes[0, 0].set_xlabel('Predicted Age')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Box plot
    axes[0, 1].boxplot(valid_results['predicted_age'])
    axes[0, 1].set_title('Age Distribution Box Plot')
    axes[0, 1].set_ylabel('Predicted Age')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Individual predictions
    x_pos = range(len(valid_results))
    axes[1, 0].bar(x_pos, valid_results['predicted_age'], alpha=0.7, color='coral')
    axes[1, 0].set_title('Individual Predictions')
    axes[1, 0].set_xlabel('Sample Index')
    axes[1, 0].set_ylabel('Predicted Age')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Statistics text
    ages = valid_results['predicted_age']
    stats_text = f"""
    Statistics:
    Mean: {ages.mean():.1f} years
    Median: {ages.median():.1f} years
    Std Dev: {ages.std():.1f} years
    Min: {ages.min():.1f} years
    Max: {ages.max():.1f} years
    Count: {len(ages)} images
    """
    
    axes[1, 1].text(0.1, 0.5, stats_text, transform=axes[1, 1].transAxes, 
                    fontsize=12, verticalalignment='center',
                    bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    axes[1, 1].set_title('Summary Statistics')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Run visualization if results exist
if 'results_df' in locals() and not results_df.empty:
    print("Generating visualizations...")
    visualize_age_predictions(results_df)
else:
    print("No results to visualize. Run the results generation cell first.")